# GLANCE Palm Cohort  Scabies & Healthy Baseline (Notebook 3)

## Clinical Objective
Curate laboratory/clinically-confirmed Scabies cases isolated to hand/palm/finger
presentation, paired with a genuine healthy palm control baseline, for the
GLANCE Palm Cohort.

## Datasets Used

| Source | Link | Used for |
|---|---|---|
| Scabies_Dataset_DIB (original) | *[https://data.mendeley.com/datasets/6scrcbtzbx/1]* | Scabies + Healthy base |
| Scabies Benchmark Dataset v2 | *[https://data.mendeley.com/datasets/mg6sb9b8x4/1]* | Scabies + Healthy (deduped supplement) |
| SCIN Dataset (Google/EkaCare) | https://huggingface.co/datasets/ekacare/SCIN | Scabies  manually confirmed hand/palm cases |
| SkinDisNet | https://data.mendeley.com/datasets/yj3md44hxg/2 | Scabies  metadata confirmed hand/palm/finger cases (real + augmented) |
| Human Palm and Gloves Dataset (DataCluster Labs) | https://www.kaggle.com/datasets/dataclusterlabs/palm-and-gloves-dataset | Healthy palm (mobile-captured) |
| Human Palm Images (Redeemers University) | https://www.kaggle.com/datasets/feyiamujo/human-palm-images | Healthy palm |

## Workflow Summary
1. **Base Merge**: Deduplicated original + v2 Scabies/Healthy sources (image-hash based), merged into a base set.
2. **SCIN Supplement**: Manually verified 57 real hand/palm Scabies cases from SCIN.
3. **SkinDisNet Supplement**: Used real clinical metadata (`Leision_location` field) to isolate 238 Hand/Palm/Finger-confirmed Scabies cases  no filename guessing needed. Added up to 2 augmented variants per confirmed original (traceable via `origin_image_id`) to reach volume.
4. **Healthy Balancing**: Combined DataCluster Labs (100 mobile-captured) + Redeemers University (800 DSLR-captured) with the existing base Healthy set.

## Output Volume
- `scabies`: 1,178 images
- `healthy_palm`: 1,102 images
- **Total**: 2,280 verified clinical images

In [ ]:
# Step 1: List available input datasets
from pathlib import Path
for p in Path("/kaggle/input").rglob("*"):
    if p.is_dir():
        print(p)

In [ ]:
# Step 2: Locate old (verified) and v2 dataset roots
old_root = Path("/kaggle/input/datasets/shahrozkhalid11/scabies-skin-diseases/Scabies_Dataset_DIB/Original_data")
new_root = Path("/kaggle/input/datasets/shahrozkhalid11/scabies-dataset-v2/A Benchmark Image Dataset for Scabies Detection_2./SCAB_original/SCAB_original")

print("Old root exists:", old_root.exists())
print("New (v2) root exists:", new_root.exists())

In [ ]:
# Step 3: Dedupe v2 Scabies vs old, then merge (fixed - hash computed once per image)
from PIL import Image
import imagehash

def get_images(folder):
    return [p for p in Path(folder).rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"}]

old_scabies = get_images(old_root / "Scabies")
new_scabies = get_images(new_root / "Scabies")

old_hashes = {imagehash.phash(Image.open(p)): p for p in old_scabies}
new_hashes = {p: imagehash.phash(Image.open(p)) for p in new_scabies}  # compute once

new_scabies_unique = [
    p for p, h in new_hashes.items()
    if not any(h - old_h <= 5 for old_h in old_hashes)
]

print(f"Old Scabies: {len(old_scabies)} | New unique Scabies: {len(new_scabies_unique)}")

In [ ]:
# Step 4: Dedupe v2 Healthy vs old, then merge (fixed - hash computed once per image)
old_healthy = get_images(old_root / "healthy")
new_healthy = get_images(new_root / "healthy")

old_healthy_hashes = {imagehash.phash(Image.open(p)): p for p in old_healthy}
new_healthy_hashes = {p: imagehash.phash(Image.open(p)) for p in new_healthy}  # compute once

new_healthy_unique = [
    p for p, h in new_healthy_hashes.items()
    if not any(h - old_h <= 5 for old_h in old_healthy_hashes)
]

print(f"Old Healthy: {len(old_healthy)} | New unique Healthy: {len(new_healthy_unique)}")

In [ ]:
# Step 5: Load SCIN, confirmed hand-scabies selection
from datasets import load_dataset

ds = load_dataset("ekacare/SCIN", split="train").to_pandas()

def has_scabies(row):
    return 'Scabies' in str(row.get('weighted_skin_condition_label', '')) or \
           'Scabies' in str(row.get('dermatologist_skin_condition_on_label_name', ''))

scabies_df = ds[ds.apply(has_scabies, axis=1)]

# Same split used originally: pre-filtered hand cases vs remaining
hand_parts = {'PALM', 'BACK_OF_HAND'}
def is_hand(row):
    parts = row.get('body_parts', [])
    return parts is not None and any(p in hand_parts for p in parts)

hand_scabies_df = scabies_df[scabies_df.apply(is_hand, axis=1)]
remaining_df = scabies_df[~scabies_df['case_id'].isin(hand_scabies_df['case_id'])]

# Rebuild the same ordered candidate list as before (batch1 then batch2)
candidates = []
for _, row in hand_scabies_df.iterrows():
    for i in range(len(row['images'])):
        candidates.append((row, i))
for _, row in remaining_df.iterrows():
    for i in range(len(row['images'])):
        candidates.append((row, i))

# Apply the previously-confirmed keep list
keep_spec = "0 to 8, 13 to 17, 19, 25 to 41, 50, 78 to 82, 89 to 92, 98, 118, 127 to 129, 139 to 143, 149 to 153"
keep_indices = set()
for part in keep_spec.split(","):
    part = part.strip()
    if "to" in part:
        a, b = part.split("to")
        keep_indices.update(range(int(a), int(b) + 1))
    else:
        keep_indices.add(int(part))

scin_confirmed = [candidates[i] for i in keep_indices if i < len(candidates)]
print(f"SCIN confirmed hand-scabies images: {len(scin_confirmed)}")

In [ ]:
# Step 6: Merge all sources, verify, standardize, save
import shutil, io
from PIL import Image
import pandas as pd

OUTPUT_BASE = Path("/kaggle/working/GLANCE_scabies_healthy")
(OUTPUT_BASE / "scabies").mkdir(parents=True, exist_ok=True)
(OUTPUT_BASE / "healthy_palm").mkdir(parents=True, exist_ok=True)

records = []

def save_image(img, dest_dir, prefix, idx, source_name):
    dest_name = f"{prefix}_{idx:04d}.jpg"
    dest_path = OUTPUT_BASE / dest_dir / dest_name
    img.convert("RGB").save(dest_path, "JPEG")
    w, h = img.size
    records.append({
        "file_name": dest_name,
        "file_path": str(dest_path),
        "clinical_condition": dest_dir,
        "source_dataset": source_name,
        "body_location": "palm_hand",
        "width": w, "height": h,
        "is_valid_image": True
    })

idx = 0
for p in old_scabies + new_scabies_unique:
    save_image(Image.open(p), "scabies", "scabies", idx, "scabies_dib_v1_v2")
    idx += 1
for row, i in scin_confirmed:
    img_data = row['images'][i]
    img = Image.open(io.BytesIO(img_data['bytes'])) if isinstance(img_data, dict) else img_data
    save_image(img, "scabies", "scabies", idx, "SCIN")
    idx += 1

idx = 0
for p in old_healthy + new_healthy_unique:
    save_image(Image.open(p), "healthy_palm", "healthy", idx, "scabies_dib_v1_v2")
    idx += 1

df_final = pd.DataFrame(records)
df_final.to_csv("/kaggle/working/scabies_healthy_standardized.csv", index=False)

print("=== FINAL RESULT ===")
print(df_final['clinical_condition'].value_counts())

In [ ]:
# Step 7: Explore SkinDisNet folder structure
from pathlib import Path

skindisnet_root = Path("/kaggle/input/datasets/shahrozkhalid11/scabies-v3")
for p in skindisnet_root.rglob("*"):
    if p.is_dir():
        print(p)

In [ ]:
# Step 8: Locate and inspect metadata CSV
from pathlib import Path
import pandas as pd

skindisnet_root = Path("/kaggle/input/datasets/shahrozkhalid11/scabies-v3")
csv_files = list(skindisnet_root.rglob("*.csv"))
print("CSV files found:", csv_files)

if csv_files:
    df_meta = pd.read_csv(csv_files[0])
    print(df_meta.columns.tolist())
    print(df_meta.head())
else:
    print("No CSV found — metadata may not have been included in this upload, or is named differently. Checking for any non-image files:")
    for p in skindisnet_root.rglob("*"):
        if p.is_file() and p.suffix.lower() not in {'.jpg', '.jpeg', '.png'}:
            print(p)

In [ ]:
# Step 9: Reload metadata, check location breakdown for Scabies
import pandas as pd
from pathlib import Path

skindisnet_root = Path("/kaggle/input/datasets/shahrozkhalid11/scabies-v3")
meta_csv = skindisnet_root / "SkinDisNet A Multi-Class Clinical Images and Metad/SkinDisNet_2/SkinDisNet_Metadata.csv"
df_meta = pd.read_csv(meta_csv)

scabies_meta = df_meta[df_meta['Diagnosis'] == 'Scabies']
print(f"Total Scabies rows in metadata: {len(scabies_meta)}")
print(scabies_meta['Leision_location'].value_counts())

In [ ]:
# Step 10: Extract Scabies images tagged Hand/Palm/Finger
df_meta['Leision_location_clean'] = df_meta['Leision_location'].str.strip()

hand_keywords = ['Hand', 'Palm', 'Finger']
scabies_hand = df_meta[
    (df_meta['Diagnosis'] == 'Scabies') &
    (df_meta['Leision_location_clean'].apply(lambda x: any(k in x for k in hand_keywords)))
]

print(f"Scabies hand/palm/finger confirmed: {len(scabies_hand)}")
print(scabies_hand['Leision_location_clean'].value_counts())

In [ ]:
# Step 11: Match metadata rows to actual image files
from PIL import Image
import shutil

scabies_dir = skindisnet_root / "SkinDisNet A Multi-Class Clinical Images and Metad/SkinDisNet_2/Preprocessed/Scabies (SC)"
all_scabies_files = list(scabies_dir.glob("*"))
print(f"Total image files in folder: {len(all_scabies_files)}")
print("Sample filenames:", [p.name for p in all_scabies_files[:5]])
print("Sample Image_id values:", scabies_hand['Image_id'].head().tolist())

In [ ]:
# Step 12: Copy confirmed hand/palm/finger Scabies images
import shutil
from PIL import Image

skindisnet_scabies_verified = []

for _, row in scabies_hand.iterrows():
    src_path = scabies_dir / f"{row['Image_id']}.jpg"
    if not src_path.exists():
        continue
    try:
        with Image.open(src_path) as img:
            img.verify()
        skindisnet_scabies_verified.append(src_path)
    except Exception:
        continue

print(f"Verified SkinDisNet hand/palm/finger Scabies images: {len(skindisnet_scabies_verified)}")

In [ ]:
# Step 13: Dedupe SkinDisNet Scabies vs already-merged Scabies
import imagehash

existing_scabies_dir = Path("/kaggle/working/GLANCE_scabies_healthy/scabies")
existing_files = list(existing_scabies_dir.glob("*"))
existing_hashes = {imagehash.phash(Image.open(p)): p for p in existing_files}

skindisnet_hashes = {p: imagehash.phash(Image.open(p)) for p in skindisnet_scabies_verified}

skindisnet_unique = [
    p for p, h in skindisnet_hashes.items()
    if not any(h - eh <= 5 for eh in existing_hashes)
]

print(f"Existing merged Scabies: {len(existing_files)}")
print(f"SkinDisNet new unique: {len(skindisnet_unique)}")

In [ ]:
# Step 14: Append SkinDisNet unique images to final dataset
df_final = pd.read_csv("/kaggle/working/scabies_healthy_standardized.csv")

start_idx = len(existing_files)
new_records = []

for i, p in enumerate(skindisnet_unique):
    idx = start_idx + i
    dest_name = f"scabies_{idx:04d}.jpg"
    dest_path = existing_scabies_dir / dest_name
    Image.open(p).convert("RGB").save(dest_path, "JPEG")
    w, h = Image.open(p).size
    new_records.append({
        "file_name": dest_name,
        "file_path": str(dest_path),
        "clinical_condition": "scabies",
        "source_dataset": "SkinDisNet",
        "body_location": "palm_hand",
        "width": w, "height": h,
        "is_valid_image": True
    })

df_final = pd.concat([df_final, pd.DataFrame(new_records)], ignore_index=True)
df_final.to_csv("/kaggle/working/scabies_healthy_standardized.csv", index=False)

print("=== UPDATED FINAL RESULT ===")
print(df_final['clinical_condition'].value_counts())

In [ ]:
# Step 15: Check if Augmented folder has matching hand/palm scabies images
scabies_aug_dir = skindisnet_root / "SkinDisNet A Multi-Class Clinical Images and Metad/SkinDisNet_2/Augmented/Scabies (SC)"
aug_files = list(scabies_aug_dir.glob("*"))
print(f"Total augmented Scabies images: {len(aug_files)}")
print("Sample filenames:", [p.name for p in aug_files[:10]])

In [ ]:
# Step 16: Pull augmented scabies images from hand/palm-confirmed originals only
import re

confirmed_ids = set(scabies_hand['Image_id'].tolist())  # e.g. 'SC (170)'

def base_id(fname):
    m = re.match(r"(SC \(\d+\))_\d+\.jpg", fname)
    return m.group(1) if m else None

aug_hand_files = [p for p in aug_files if base_id(p.name) in confirmed_ids]
print(f"Augmented images from hand/palm-confirmed originals: {len(aug_hand_files)}")

In [ ]:
# Step 17: Cap augmented selection to 2 per original image
from collections import defaultdict

grouped = defaultdict(list)
for p in aug_hand_files:
    grouped[base_id(p.name)].append(p)

selected_aug = []
for bid, files in grouped.items():
    selected_aug.extend(files[:2])  # take max 2 per original

print(f"Selected augmented images (max 2 per original): {len(selected_aug)}")
print(f"Projected new Scabies total: {702 + len(selected_aug)}")

In [ ]:
# Step 18: Copy selected augmented images into final dataset
df_final = pd.read_csv("/kaggle/working/scabies_healthy_standardized.csv")

start_idx = len(df_final[df_final['clinical_condition'] == 'scabies'])
new_records = []

for i, p in enumerate(selected_aug):
    idx = start_idx + i
    dest_name = f"scabies_{idx:04d}.jpg"
    dest_path = existing_scabies_dir / dest_name
    try:
        with Image.open(p) as img:
            img.verify()
        img = Image.open(p)
        img.convert("RGB").save(dest_path, "JPEG")
        w, h = img.size
    except Exception:
        continue

    new_records.append({
        "file_name": dest_name,
        "file_path": str(dest_path),
        "clinical_condition": "scabies",
        "source_dataset": "SkinDisNet_augmented",
        "origin_image_id": base_id(p.name),
        "body_location": "palm_hand",
        "width": w, "height": h,
        "is_valid_image": True
    })

df_final = pd.concat([df_final, pd.DataFrame(new_records)], ignore_index=True)
df_final.to_csv("/kaggle/working/scabies_healthy_standardized.csv", index=False)

print("=== FINAL RESULT ===")
print(df_final['clinical_condition'].value_counts())

In [ ]:
# Step 19: Locate attached Palm & Gloves dataset, count images
from pathlib import Path

palm_gloves_root = Path("/kaggle/input/human-palm-and-gloves-dataset-human-...")  # adjust to exact folder name shown in sidebar
for p in Path("/kaggle/input").rglob("*"):
    if p.is_dir() and "palm" in p.name.lower() and "gloves" in p.name.lower():
        print(p)

In [ ]:
# Step 20: Count images in Palm & Gloves dataset
from pathlib import Path

palm_gloves_root = Path("/kaggle/input/datasets/dataclusterlabs/palm-and-gloves-dataset")
all_imgs = [p for p in palm_gloves_root.rglob("*") if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}]

print(f"Total images found: {len(all_imgs)}")
for p in palm_gloves_root.rglob("*"):
    if p.is_dir():
        print(p)

In [ ]:
# Step 21: Download Redeemers University Human Palm Images dataset
import kagglehub

redeemers_path = kagglehub.dataset_download("feyiamujo/human-palm-images")
print(f"Downloaded to: {redeemers_path}")

In [ ]:
# Step 22: Count images
from pathlib import Path

redeemers_root = Path(redeemers_path)
all_imgs = [p for p in redeemers_root.rglob("*") if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}]
print(f"Total images found: {len(all_imgs)}")

In [ ]:
# Step 23: Merge all healthy palm sources into final dataset
from PIL import Image
import pandas as pd
from pathlib import Path

OUTPUT_BASE = Path("/kaggle/working/GLANCE_scabies_healthy")
healthy_dir = OUTPUT_BASE / "healthy_palm"

df_final = pd.read_csv("/kaggle/working/scabies_healthy_standardized.csv")
start_idx = len(df_final[df_final['clinical_condition'] == 'healthy_palm'])

new_records = []
idx = start_idx

# Source 1: Palm & Gloves (100 images)
palm_gloves_imgs = [p for p in palm_gloves_root.rglob("*") if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}]
for p in palm_gloves_imgs:
    try:
        img = Image.open(p)
        img.verify()
        img = Image.open(p)
        dest_name = f"healthy_{idx:04d}.jpg"
        dest_path = healthy_dir / dest_name
        img.convert("RGB").save(dest_path, "JPEG")
        w, h = img.size
        new_records.append({
            "file_name": dest_name, "file_path": str(dest_path),
            "clinical_condition": "healthy_palm", "source_dataset": "palm_and_gloves_dataclusterlabs",
            "body_location": "palm_hand", "width": w, "height": h, "is_valid_image": True
        })
        idx += 1
    except Exception:
        continue

# Source 2: Redeemers University (800 images)
redeemers_imgs = [p for p in redeemers_root.rglob("*") if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}]
for p in redeemers_imgs:
    try:
        img = Image.open(p)
        img.verify()
        img = Image.open(p)
        dest_name = f"healthy_{idx:04d}.jpg"
        dest_path = healthy_dir / dest_name
        img.convert("RGB").save(dest_path, "JPEG")
        w, h = img.size
        new_records.append({
            "file_name": dest_name, "file_path": str(dest_path),
            "clinical_condition": "healthy_palm", "source_dataset": "redeemers_university",
            "body_location": "palm_hand", "width": w, "height": h, "is_valid_image": True
        })
        idx += 1
    except Exception:
        continue

df_final = pd.concat([df_final, pd.DataFrame(new_records)], ignore_index=True)
df_final.to_csv("/kaggle/working/scabies_healthy_standardized.csv", index=False)

print("=== FINAL RESULT ===")
print(df_final['clinical_condition'].value_counts())

In [ ]:
# Step 24: Clean up intermediate files, keep only final dataset
import shutil
from pathlib import Path

working = Path("/kaggle/working")

to_remove = [
    working / "roboflow_scabies",
    working / "hand_landmarker.task",
    working / "hand_detection_results.csv",
    working / "index_map.json",
    working / "numbered_grid.png",
]

for item in to_remove:
    if item.exists():
        shutil.rmtree(item) if item.is_dir() else item.unlink()
        print(f"Removed: {item}")
    else:
        print(f"Not found (already clean): {item}")

print("\n=== Remaining in /kaggle/working ===")
for p in working.iterdir():
    print(" -", p)

In [ ]:
# Step 25: Show random sample with class labels
import matplotlib.pyplot as plt
from PIL import Image

sample = df_final.sample(9).reset_index(drop=True)

fig, axes = plt.subplots(3, 3, figsize=(14, 14))
for ax, (_, row) in zip(axes.flat, sample.iterrows()):
    img = Image.open(row['file_path'])
    ax.imshow(img)
    color = 'crimson' if row['clinical_condition'] == 'scabies' else 'seagreen'
    ax.set_title(row['clinical_condition'].replace('_', ' ').title(),
                 fontsize=14, fontweight='bold', color=color)
    ax.axis('off')

plt.tight_layout()
plt.show()